# Static UMAP Metric Run Script Generator

This notebook is a Delta control panel for the static UMAP metric workflow. It intentionally delegates the real work to the maintained scripts:

1. `static_umap_metric_job_generator.py` creates and submits Slurm job scripts.
2. `static_umap_metrics.py` runs one run/experiment analysis job and writes plots, metric CSVs, summary CSVs, and JSON.

The workflow uses `catalog2.fits` through `catalog-key all`. It does not use separate xmatch catalogs or mmfs paths.

In [ ]:
from __future__ import annotations

import shlex
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "static_umap_metric_job_generator.py").exists():
    candidate = NOTEBOOK_DIR / "Hyrax-Research"
    if (candidate / "static_umap_metric_job_generator.py").exists():
        NOTEBOOK_DIR = candidate

GENERATOR = NOTEBOOK_DIR / "static_umap_metric_job_generator.py"
ANALYSIS_SCRIPT = NOTEBOOK_DIR / "static_umap_metrics.py"

if not GENERATOR.exists():
    raise FileNotFoundError(f"Could not find generator script at {GENERATOR}")
if not ANALYSIS_SCRIPT.exists():
    raise FileNotFoundError(f"Could not find analysis script at {ANALYSIS_SCRIPT}")


def run_command(cmd, *, check=True):
    cmd = [str(part) for part in cmd]
    print("$", shlex.join(cmd))
    result = subprocess.run(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    if result.stdout:
        print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Generator: {GENERATOR}")
print(f"Analysis script: {ANALYSIS_SCRIPT}")

## Configure the batch

The defaults below match the current Delta workflow. Leave `RUN_EXPTS` and `OVERLAY_GROUPS` empty to use the generator's built-in notebook plan: Run 10 gets `time_since_merger` plus `future_merger_flags`, and Run 11 gets `time_since_merger`.

In [ ]:
PROFILE = "delta"
BASE_DIR = Path("/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs")
# OUTPUT_DIR = BASE_DIR / "static_umap_metrics"
OUTPUT_DIR = BASE_DIR / "static_umap_metrics_tsm_le_1p5gyr"  # For the time_since_merger <= [Whatever number, ex. 1.5] Gyr run

GENERATOR_PYTHON = "python"
JOB_PYTHON = "python"
CATALOG_KEY = "all"
JOB_PREFIX = "plot_metrics_tsm_1p5_"

# Leave empty to use the generator's default Run 10 / Run 11 plan.
# If you add values here, each string should look like "10:7,10,12" or "11:7-14".
RUN_EXPTS = ["10:1-18"] 

# Leave empty to use the generator's per-run defaults. If set, these groups apply to every explicit RUN_EXPTS entry.
OVERLAY_GROUPS = ["time_since_merger"]
N_PERMUTATIONS = 500
MIN_CLUSTER_SIZE = 15
SEED = 42
DPI = 150
TIME_SINCE_MERGER_MAX_GYR = 1.5    # 0.5, 0.75, 1.0, 1.25, 1.5 Gyr
INCLUDE_HIGHDIM = False     # Keep OFF for normal 2D sweeps; HD jobs below override this safely
REQUIRE_HIGHDIM = False     # Use True only with INCLUDE_HIGHDIM; HD notebook jobs default to requiring HD

DENSITY = False
LOG_COLORBAR = False
SHOW_LEGEND = True
SUPPRESS_LOGS = True

# Example: {"partition": "cpu", "mem": "64G", "time": "4:00:00"}
SLURM_OVERRIDES = {}

# None uses the generator defaults. [] removes setup lines. A list replaces setup lines.
SETUP_LINES = None
NO_DEFAULT_SETUP = False


def build_generator_command(action, *, dry_run=False):
    cmd = [
        GENERATOR_PYTHON,
        GENERATOR,
        action,
        "--profile",
        PROFILE,
        "--base-directory",
        BASE_DIR,
        "--output-dir",
        OUTPUT_DIR,
        "--analysis-script",
        ANALYSIS_SCRIPT,
        "--python-executable",
        JOB_PYTHON,
        "--job-prefix",
        JOB_PREFIX,
        "--catalog-key",
        CATALOG_KEY,
        "--n-permutations",
        N_PERMUTATIONS,
        "--min-cluster-size",
        MIN_CLUSTER_SIZE,
        "--seed",
        SEED,
        "--dpi",
        DPI,
    ]

    for spec in RUN_EXPTS:
        cmd.extend(["--run-expts", spec])
    for group in OVERLAY_GROUPS:
        cmd.extend(["--overlay-group", group])
    for key, value in SLURM_OVERRIDES.items():
        cmd.extend(["--slurm", f"{key}={value}"])

    if TIME_SINCE_MERGER_MAX_GYR is not None:
        cmd.extend(["--time-since-merger-max-gyr", TIME_SINCE_MERGER_MAX_GYR])
    if INCLUDE_HIGHDIM:
        cmd.append("--include-highdim")
    if REQUIRE_HIGHDIM:
        cmd.append("--require-highdim")
    if DENSITY:
        cmd.append("--density")
    if LOG_COLORBAR:
        cmd.append("--log-colorbar")
    if not SHOW_LEGEND:
        cmd.append("--no-show-legend")
    if not SUPPRESS_LOGS:
        cmd.append("--no-suppress-logs")

    if SETUP_LINES is not None:
        if len(SETUP_LINES) == 0:
            cmd.append("--no-default-setup")
        else:
            for line in SETUP_LINES:
                cmd.extend(["--setup-line", line])
    elif NO_DEFAULT_SETUP:
        cmd.append("--no-default-setup")

    if dry_run:
        cmd.append("--dry-run")
    return cmd

print(f"Profile: {PROFILE}")
print(f"Base directory: {BASE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Catalog key: {CATALOG_KEY}")


## Preview the plan

Run this before creating scripts. It prints the runs, experiments, overlay groups, Slurm settings, catalog key, and output path.

In [ ]:
run_command(build_generator_command("print-plan"))

## Generate Slurm scripts

This writes `plot_metrics<run>_<expt>.sh` into each selected `run<run>/` directory. It does not submit anything.

In [ ]:
run_command(build_generator_command("write"))

## Dry-run submission

This checks the generated scripts and prints the exact `sbatch` commands without submitting jobs.

In [ ]:
run_command(build_generator_command("submit-existing", dry_run=True))

## Submit existing scripts

The guard below prevents accidental submissions. Set `SUBMIT_JOBS = True` only after the dry-run output looks correct.

In [ ]:
SUBMIT_JOBS = False

if SUBMIT_JOBS:
    run_command(build_generator_command("submit-existing"))
else:
    print("SUBMIT_JOBS is False; no jobs submitted.")


## View multi-cutoff results

This section scans cutoff-specific output folders under `BASE_DIR`, combines every metrics CSV for a chosen run, ranks the strongest run/expt/cutoff/overlay rows, and displays summary tables plus optional PNGs. It can scan both the original 2D result folders and HD result folders ending in `_hd`.

Change `RANKING_MODE` to switch the sort logic:

- `mnln_p`: lowest 2D MNLN p-value, then lowest 2D MNLN ratio.
- `mnln_effect`: lowest 2D MNLN ratio, then lowest 2D MNLN p-value.
- `cmc_p`: lowest 2D CMC-Gini p-value, then highest 2D CMC-Gini.
- `cmc_effect`: highest 2D CMC-Gini z-score, then lowest 2D CMC-Gini p-value.
- `composite`: average rank across 2D MNLN p-value, 2D MNLN ratio, 2D CMC p-value, and 2D CMC z-score.
- `stable_cutoff`: groups by result family, run, experiment, and overlay, then ranks cases that stay strong across multiple cutoffs.
- `mnln_hd_p`, `mnln_hd_effect`, `cmc_hd_p`, `cmc_hd_effect`, `hd_composite`: HD equivalents, only meaningful for rows with HD metrics.
- `2d_hd_agreement`: ranks rows that are strong in both 2D and HD metrics.

Metric reminder: p-values are not the probability that clusters exist. Lower p-values mean a random same-sized selected set would less often look at least this clustered by that metric. For MNLN, ratios below 1 mean selected points are closer together than random. For CMC-Gini, higher Gini/z-score means selected labels concentrate more strongly in HDBSCAN regions.


In [ ]:
import math
import re

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)


RESULT_RUN = 10
RESULT_CUTOFFS = [0.5, 0.75, 1.0, 1.25, 1.5]
RESULT_FAMILIES = ["2d", "hd"]       # Use ["2d"] for original results only or ["hd"] for HD-only viewing.
RESULT_OUTPUT_GLOB = "static_umap_metrics_tsm_le_*gyr"  # Kept for custom/single-family scans.
RESULT_OUTPUT_GLOBS = {
    "2d": "static_umap_metrics_tsm_le_*gyr",
    "hd": "static_umap_metrics_tsm_le_*gyr_hd",
}
RANKING_MODE = "mnln_p"
TOP_N = 20

# Optional filters. Leave as None to include everything found.
RESULT_EXPTS = None                 # Example: [1, 5, 8, 12]
RESULT_OVERLAY_GROUPS = None        # Example: ["time_since_merger"]
RESULT_OVERLAY_KEYS = None          # Example: ["Major_TimeSinceMerger"]
RESULT_OVERLAY_LABEL_CONTAINS = None
MIN_MATCHED = 2

PIVOT_VALUE = "mnln_2d_p_value"
SHOW_TOP_PNGS = True
MAX_GALLERY_IMAGES = 8

STABLE_P_VALUE_MAX = 0.05
STABLE_RATIO_MAX = 1.0
STABLE_MIN_CUTOFFS = 2

HD_REQUIRED_RESULT_COLUMNS = ["mnln_hd_p_value", "mnln_hd_ratio", "cmc_hd_p_value", "cmc_hd_z_score"]

TOP_RESULT_COLUMNS = [
    "result_family",
    "has_highdim_metrics",
    "run",
    "expt",
    "cutoff_gyr",
    "overlay_label",
    "overlay_key",
    "n_matched",
    "mnln_2d_p_value",
    "mnln_2d_ratio",
    "mnln_2d_observed",
    "mnln_2d_expected",
    "cmc_2d_p_value",
    "cmc_2d_gini",
    "cmc_2d_expected_gini",
    "cmc_2d_z_score",
    "cmc_2d_n_clusters",
    "mnln_hd_p_value",
    "mnln_hd_ratio",
    "mnln_hd_observed",
    "mnln_hd_expected",
    "mnln_hd_z_score",
    "cmc_hd_p_value",
    "cmc_hd_gini",
    "cmc_hd_expected_gini",
    "cmc_hd_z_score",
    "cmc_hd_n_clusters",
    "png_path",
]

STABLE_RESULT_COLUMNS = [
    "result_family",
    "has_highdim_metrics",
    "run",
    "expt",
    "overlay_label",
    "overlay_key",
    "n_cutoffs",
    "cutoffs_seen",
    "n_mnln_p_le_threshold",
    "n_mnln_ratio_lt_threshold",
    "median_mnln_2d_p_value",
    "median_mnln_2d_ratio",
    "best_mnln_2d_p_value",
    "best_mnln_2d_ratio",
    "best_cutoff_gyr",
    "best_cmc_2d_p_value",
    "best_cmc_2d_z_score",
    "best_mnln_hd_p_value",
    "best_mnln_hd_ratio",
    "best_cmc_hd_p_value",
    "best_cmc_hd_z_score",
    "png_path",
]

NUMERIC_RESULT_COLUMNS = [
    "run",
    "expt",
    "cutoff_gyr",
    "n_matched",
    "mnln_2d_p_value",
    "mnln_2d_ratio",
    "mnln_2d_observed",
    "mnln_2d_expected",
    "mnln_2d_z_score",
    "cmc_2d_p_value",
    "cmc_2d_gini",
    "cmc_2d_expected_gini",
    "cmc_2d_z_score",
    "cmc_2d_n_clusters",
    "mnln_hd_p_value",
    "mnln_hd_ratio",
    "mnln_hd_observed",
    "mnln_hd_expected",
    "mnln_hd_z_score",
    "cmc_hd_p_value",
    "cmc_hd_gini",
    "cmc_hd_expected_gini",
    "cmc_hd_z_score",
    "cmc_hd_n_clusters",
]

RANKING_DESCRIPTIONS = {
    "mnln_p": "Lowest 2D MNLN p-value, then lowest 2D MNLN ratio.",
    "mnln_effect": "Lowest 2D MNLN ratio, then lowest 2D MNLN p-value.",
    "cmc_p": "Lowest 2D CMC-Gini p-value, then highest 2D CMC-Gini.",
    "cmc_effect": "Highest 2D CMC-Gini z-score, then lowest 2D CMC-Gini p-value.",
    "composite": "Average rank across 2D MNLN p-value, 2D MNLN ratio, 2D CMC p-value, and 2D CMC z-score.",
    "stable_cutoff": "Groups by result family, run, experiment, and overlay, then ranks cases that stay strong across cutoffs.",
    "mnln_hd_p": "Lowest HD MNLN p-value, then lowest HD MNLN ratio.",
    "mnln_hd_effect": "Lowest HD MNLN ratio, then lowest HD MNLN p-value.",
    "cmc_hd_p": "Lowest HD CMC-Gini p-value, then highest HD CMC-Gini.",
    "cmc_hd_effect": "Highest HD CMC-Gini z-score, then lowest HD CMC-Gini p-value.",
    "hd_composite": "Average rank across HD MNLN p-value, HD MNLN ratio, HD CMC p-value, and HD CMC z-score.",
    "2d_hd_agreement": "Average rank across 2D and HD MNLN/CMC evidence; rows missing HD rank last.",
}


def parse_run_expt_from_path(path):
    path = Path(path)
    run = None
    expt = None
    for part in path.parts:
        run_match = re.fullmatch(r"run(\d+)", part)
        expt_match = re.fullmatch(r"expt(\d+)", part)
        if run_match:
            run = int(run_match.group(1))
        if expt_match:
            expt = int(expt_match.group(1))
    return run, expt


def parse_cutoff_from_name(name):
    match = re.search(r"tsm_le_([0-9]+(?:[p.][0-9]+)?)gyr", str(name))
    if not match:
        return math.nan
    return float(match.group(1).replace("p", "."))


def infer_result_family(path):
    name = Path(path).name
    return "hd" if name.endswith("_hd") else "2d"


def cutoff_is_requested(cutoff, requested_cutoffs):
    if requested_cutoffs is None or len(requested_cutoffs) == 0:
        return True
    if pd.isna(cutoff):
        return False
    return any(math.isclose(float(cutoff), float(requested), rel_tol=0, abs_tol=1e-9) for requested in requested_cutoffs)


def iter_result_glob_requests(output_glob=RESULT_OUTPUT_GLOB, output_globs=RESULT_OUTPUT_GLOBS, families=RESULT_FAMILIES):
    if output_globs is None:
        yield "custom", output_glob
        return
    requested = families if families is not None else list(output_globs)
    for family in requested:
        if family not in output_globs:
            raise ValueError(f"Unknown result family {family!r}. Options: {sorted(output_globs)}")
        yield family, output_globs[family]


def discover_result_roots(
    base_dir=BASE_DIR,
    output_glob=RESULT_OUTPUT_GLOB,
    cutoffs=RESULT_CUTOFFS,
    result_families=RESULT_FAMILIES,
    output_globs=RESULT_OUTPUT_GLOBS,
):
    base_dir = Path(base_dir)
    roots = []
    seen = set()
    for family, glob_pattern in iter_result_glob_requests(output_glob=output_glob, output_globs=output_globs, families=result_families):
        for root in sorted(base_dir.glob(glob_pattern)):
            if not root.is_dir():
                continue
            root_key = str(root)
            if root_key in seen:
                continue
            seen.add(root_key)
            cutoff = parse_cutoff_from_name(root.name)
            if not cutoff_is_requested(cutoff, cutoffs):
                continue
            roots.append({
                "result_root": root,
                "cutoff_gyr": cutoff,
                "result_family": infer_result_family(root) if family == "custom" else family,
            })
    return sorted(roots, key=lambda item: (item["result_family"], pd.isna(item["cutoff_gyr"]), item["cutoff_gyr"], str(item["result_root"])))


def discover_result_dirs(
    base_dir=BASE_DIR,
    output_glob=RESULT_OUTPUT_GLOB,
    cutoffs=RESULT_CUTOFFS,
    run=RESULT_RUN,
    expts=RESULT_EXPTS,
    result_families=RESULT_FAMILIES,
    output_globs=RESULT_OUTPUT_GLOBS,
):
    expt_filter = None if expts is None else {int(expt) for expt in expts}
    rows = []
    for root_info in discover_result_roots(
        base_dir=base_dir,
        output_glob=output_glob,
        cutoffs=cutoffs,
        result_families=result_families,
        output_globs=output_globs,
    ):
        root = root_info["result_root"]
        if run is None:
            candidates = sorted(root.glob("run*/expt*"))
        else:
            candidates = sorted((root / f"run{int(run)}").glob("expt*"))
        for result_dir in candidates:
            if not result_dir.is_dir():
                continue
            parsed_run, parsed_expt = parse_run_expt_from_path(result_dir)
            if expt_filter is not None and parsed_expt not in expt_filter:
                continue
            rows.append({
                "result_root": root,
                "result_dir": result_dir,
                "cutoff_gyr": root_info["cutoff_gyr"],
                "result_family": root_info["result_family"],
                "run": parsed_run,
                "expt": parsed_expt,
            })
    return rows


def make_png_path(row):
    overlay_group = row.get("overlay_group")
    run = row.get("run")
    expt = row.get("expt")
    result_dir = Path(row.get("result_dir"))
    if pd.isna(overlay_group) or pd.isna(run) or pd.isna(expt):
        return ""
    return str(result_dir / f"run{int(run)}_expt{int(expt)}_{overlay_group}.png")


def coerce_result_columns(frame):
    frame = frame.copy()
    for column in NUMERIC_RESULT_COLUMNS:
        if column not in frame.columns:
            frame[column] = pd.NA
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    return frame


def load_multicutoff_csvs(
    pattern,
    base_dir=BASE_DIR,
    output_glob=RESULT_OUTPUT_GLOB,
    cutoffs=RESULT_CUTOFFS,
    run=RESULT_RUN,
    expts=RESULT_EXPTS,
    result_families=RESULT_FAMILIES,
    output_globs=RESULT_OUTPUT_GLOBS,
):
    frames = []
    result_dirs = discover_result_dirs(
        base_dir=base_dir,
        output_glob=output_glob,
        cutoffs=cutoffs,
        run=run,
        expts=expts,
        result_families=result_families,
        output_globs=output_globs,
    )
    for info in result_dirs:
        for path in sorted(info["result_dir"].glob(pattern)):
            frame = pd.read_csv(path)
            frame["source_file"] = str(path)
            frame["result_root"] = str(info["result_root"])
            frame["result_dir"] = str(info["result_dir"])
            frame["result_family"] = info["result_family"]
            frame["cutoff_gyr"] = info["cutoff_gyr"]
            if "run" not in frame.columns or frame["run"].isna().all():
                frame["run"] = info["run"]
            if "expt" not in frame.columns or frame["expt"].isna().all():
                frame["expt"] = info["expt"]
            frames.append(frame)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def load_multicutoff_metric_results(**kwargs):
    frame = load_multicutoff_csvs("*_metrics.csv", **kwargs)
    if frame.empty:
        return frame
    frame = coerce_result_columns(frame)
    frame["png_path"] = frame.apply(make_png_path, axis=1)
    frame["png_exists"] = frame["png_path"].map(lambda item: Path(item).exists() if item else False)
    frame["has_highdim_metrics"] = frame[HD_REQUIRED_RESULT_COLUMNS].notna().all(axis=1)
    return frame


def load_multicutoff_overlay_summary(**kwargs):
    frame = load_multicutoff_csvs("*_overlay_summary.csv", **kwargs)
    if frame.empty:
        return frame
    return coerce_result_columns(frame)


def filter_metric_results(
    frame,
    expts=RESULT_EXPTS,
    overlay_groups=RESULT_OVERLAY_GROUPS,
    overlay_keys=RESULT_OVERLAY_KEYS,
    overlay_label_contains=RESULT_OVERLAY_LABEL_CONTAINS,
    min_matched=MIN_MATCHED,
):
    if frame.empty:
        return frame.copy()
    work = frame.copy()
    if expts is not None:
        work = work[work["expt"].isin([int(expt) for expt in expts])]
    if overlay_groups is not None and "overlay_group" in work.columns:
        work = work[work["overlay_group"].isin(overlay_groups)]
    if overlay_keys is not None and "overlay_key" in work.columns:
        work = work[work["overlay_key"].isin(overlay_keys)]
    if overlay_label_contains:
        work = work[work["overlay_label"].astype(str).str.contains(overlay_label_contains, case=False, na=False)]
    if min_matched is not None and "n_matched" in work.columns:
        work = work[work["n_matched"] >= int(min_matched)]
    return work.reset_index(drop=True)


def add_rank_columns(work, rank_specs, score_column):
    rank_columns = []
    for column, ascending in rank_specs:
        rank_column = f"rank_{column}"
        work[rank_column] = work[column].rank(ascending=ascending, na_option="bottom", method="min")
        rank_columns.append(rank_column)
    work[score_column] = work[rank_columns].mean(axis=1)
    return work


def sort_job_level_results(frame, mode):
    work = frame.copy()
    if mode == "mnln_p":
        return work.sort_values(["mnln_2d_p_value", "mnln_2d_ratio"], ascending=[True, True], na_position="last")
    if mode == "mnln_effect":
        return work.sort_values(["mnln_2d_ratio", "mnln_2d_p_value"], ascending=[True, True], na_position="last")
    if mode == "cmc_p":
        return work.sort_values(["cmc_2d_p_value", "cmc_2d_gini"], ascending=[True, False], na_position="last")
    if mode == "cmc_effect":
        return work.sort_values(["cmc_2d_z_score", "cmc_2d_p_value"], ascending=[False, True], na_position="last")
    if mode == "mnln_hd_p":
        return work.sort_values(["mnln_hd_p_value", "mnln_hd_ratio"], ascending=[True, True], na_position="last")
    if mode == "mnln_hd_effect":
        return work.sort_values(["mnln_hd_ratio", "mnln_hd_p_value"], ascending=[True, True], na_position="last")
    if mode == "cmc_hd_p":
        return work.sort_values(["cmc_hd_p_value", "cmc_hd_gini"], ascending=[True, False], na_position="last")
    if mode == "cmc_hd_effect":
        return work.sort_values(["cmc_hd_z_score", "cmc_hd_p_value"], ascending=[False, True], na_position="last")
    if mode == "composite":
        work = add_rank_columns(
            work,
            [
                ("mnln_2d_p_value", True),
                ("mnln_2d_ratio", True),
                ("cmc_2d_p_value", True),
                ("cmc_2d_z_score", False),
            ],
            "composite_rank_score",
        )
        return work.sort_values(["composite_rank_score", "mnln_2d_p_value", "mnln_2d_ratio"], ascending=[True, True, True], na_position="last")
    if mode == "hd_composite":
        work = add_rank_columns(
            work,
            [
                ("mnln_hd_p_value", True),
                ("mnln_hd_ratio", True),
                ("cmc_hd_p_value", True),
                ("cmc_hd_z_score", False),
            ],
            "hd_composite_rank_score",
        )
        return work.sort_values(["hd_composite_rank_score", "mnln_hd_p_value", "mnln_hd_ratio"], ascending=[True, True, True], na_position="last")
    if mode == "2d_hd_agreement":
        work = add_rank_columns(
            work,
            [
                ("mnln_2d_p_value", True),
                ("mnln_2d_ratio", True),
                ("cmc_2d_p_value", True),
                ("cmc_2d_z_score", False),
                ("mnln_hd_p_value", True),
                ("mnln_hd_ratio", True),
                ("cmc_hd_p_value", True),
                ("cmc_hd_z_score", False),
            ],
            "agreement_rank_score",
        )
        return work.sort_values(["agreement_rank_score", "mnln_hd_p_value", "mnln_2d_p_value"], ascending=[True, True, True], na_position="last")
    raise ValueError(f"Unknown RANKING_MODE {mode!r}. Options: {', '.join(RANKING_DESCRIPTIONS)}")


def format_cutoff_list(values):
    clean = [float(value) for value in values if not pd.isna(value)]
    return ", ".join(f"{value:g}" for value in sorted(set(clean)))


def rank_stable_cutoff_results(frame):
    if frame.empty:
        return frame.copy()
    rows = []
    group_cols = ["result_family", "run", "expt", "overlay_group", "overlay_key"]
    for keys, group in frame.groupby(group_cols, dropna=False):
        n_cutoffs = int(group["cutoff_gyr"].nunique(dropna=True))
        if STABLE_MIN_CUTOFFS is not None and n_cutoffs < int(STABLE_MIN_CUTOFFS):
            continue
        best = sort_job_level_results(group, "mnln_p").iloc[0]
        row = dict(zip(group_cols, keys))
        row.update({
            "overlay_label": best.get("overlay_label"),
            "has_highdim_metrics": bool(group.get("has_highdim_metrics", pd.Series(False, index=group.index)).any()),
            "n_cutoffs": n_cutoffs,
            "cutoffs_seen": format_cutoff_list(group["cutoff_gyr"]),
            "n_mnln_p_le_threshold": int((group["mnln_2d_p_value"] <= STABLE_P_VALUE_MAX).sum()),
            "n_mnln_ratio_lt_threshold": int((group["mnln_2d_ratio"] < STABLE_RATIO_MAX).sum()),
            "median_mnln_2d_p_value": group["mnln_2d_p_value"].median(),
            "median_mnln_2d_ratio": group["mnln_2d_ratio"].median(),
            "best_mnln_2d_p_value": group["mnln_2d_p_value"].min(),
            "best_mnln_2d_ratio": group["mnln_2d_ratio"].min(),
            "best_cutoff_gyr": best.get("cutoff_gyr"),
            "best_cmc_2d_p_value": group["cmc_2d_p_value"].min(),
            "best_cmc_2d_z_score": group["cmc_2d_z_score"].max(),
            "best_mnln_hd_p_value": group["mnln_hd_p_value"].min(),
            "best_mnln_hd_ratio": group["mnln_hd_ratio"].min(),
            "best_cmc_hd_p_value": group["cmc_hd_p_value"].min(),
            "best_cmc_hd_z_score": group["cmc_hd_z_score"].max(),
            "n_matched": best.get("n_matched"),
            "mnln_2d_p_value": best.get("mnln_2d_p_value"),
            "mnln_2d_ratio": best.get("mnln_2d_ratio"),
            "cmc_2d_p_value": best.get("cmc_2d_p_value"),
            "cmc_2d_gini": best.get("cmc_2d_gini"),
            "cmc_2d_z_score": best.get("cmc_2d_z_score"),
            "mnln_hd_p_value": best.get("mnln_hd_p_value"),
            "mnln_hd_ratio": best.get("mnln_hd_ratio"),
            "cmc_hd_p_value": best.get("cmc_hd_p_value"),
            "cmc_hd_gini": best.get("cmc_hd_gini"),
            "cmc_hd_z_score": best.get("cmc_hd_z_score"),
            "result_dir": best.get("result_dir"),
            "source_file": best.get("source_file"),
            "png_path": best.get("png_path"),
            "png_exists": best.get("png_exists", False),
        })
        rows.append(row)
    if not rows:
        return pd.DataFrame()
    stable = pd.DataFrame(rows)
    return stable.sort_values(
        [
            "n_mnln_p_le_threshold",
            "n_mnln_ratio_lt_threshold",
            "median_mnln_2d_p_value",
            "median_mnln_2d_ratio",
            "best_mnln_2d_p_value",
        ],
        ascending=[False, False, True, True, True],
        na_position="last",
    ).reset_index(drop=True)


def rank_metric_results(frame, mode=RANKING_MODE):
    if frame.empty:
        return frame.copy()
    if mode == "stable_cutoff":
        ranked = rank_stable_cutoff_results(frame)
    else:
        ranked = sort_job_level_results(frame, mode).reset_index(drop=True)
    if not ranked.empty:
        ranked.insert(0, "rank", range(1, len(ranked) + 1))
    return ranked


def select_columns(frame, columns):
    available = [column for column in columns if column in frame.columns]
    return frame.loc[:, available]


def display_table(title, frame, columns=None, max_rows=TOP_N):
    display(Markdown(f"### {title}"))
    if frame.empty:
        print("No rows found.")
        return
    shown = frame if max_rows is None else frame.head(max_rows)
    if columns is not None:
        shown = select_columns(shown, ["rank"] + columns if "rank" in shown.columns and "rank" not in columns else columns)
    display(shown)


def best_per_group(frame, group_cols, mode=RANKING_MODE):
    if frame.empty:
        return frame.copy()
    ranked = sort_job_level_results(frame, "mnln_p" if mode == "stable_cutoff" else mode).copy()
    return ranked.groupby(group_cols, dropna=False, as_index=False).head(1).reset_index(drop=True)


def make_cutoff_expt_pivot(frame, value=PIVOT_VALUE):
    if frame.empty or value not in frame.columns:
        return pd.DataFrame()
    return frame.pivot_table(index="expt", columns="cutoff_gyr", values=value, aggfunc="min").sort_index()


def display_highdim_coverage(frame):
    display(Markdown("### High-dimensional coverage"))
    if frame.empty:
        print("No metric rows found.")
        return
    total = len(frame)
    hd_rows = int(frame.get("has_highdim_metrics", pd.Series(False, index=frame.index)).sum())
    print(f"Rows with HD metrics: {hd_rows} of {total}")
    if "result_family" in frame.columns:
        family_counts = frame.groupby("result_family")["has_highdim_metrics"].agg(rows="size", rows_with_hd="sum").reset_index()
        display(family_counts)
        hd_family = frame[frame["result_family"].eq("hd")]
        missing_dirs = sorted(hd_family.loc[~hd_family["has_highdim_metrics"], "result_dir"].dropna().astype(str).unique())
        if missing_dirs:
            print("HD result folders with CSV rows but missing HD metric columns:")
            for path in missing_dirs[:20]:
                print(f"  {path}")
            if len(missing_dirs) > 20:
                print(f"  ... {len(missing_dirs) - 20} more")


def display_png_gallery(frame, max_images=MAX_GALLERY_IMAGES):
    if frame.empty or max_images == 0:
        return
    png_rows = frame[frame.get("png_path", "").astype(str).map(lambda item: bool(item) and Path(item).exists())]
    if png_rows.empty:
        print("No PNG files found for the displayed top rows.")
        return
    display(Markdown(f"### Top PNGs ({min(len(png_rows), max_images)} of {len(png_rows)})"))
    for _, row in png_rows.head(max_images).iterrows():
        cutoff = row.get("cutoff_gyr", math.nan)
        if pd.isna(cutoff):
            cutoff = row.get("best_cutoff_gyr", math.nan)
        title = (
            f"{row.get('result_family', 'result')} | Run {int(row['run'])}, Expt {int(row['expt'])}, cutoff {cutoff:g} Gyr, "
            f"{row.get('overlay_label', row.get('overlay_key', 'overlay'))}"
        )
        metric_text = (
            f"2D MNLN p={row.get('mnln_2d_p_value', math.nan):.4g}, "
            f"2D ratio={row.get('mnln_2d_ratio', math.nan):.4g}, "
            f"HD MNLN p={row.get('mnln_hd_p_value', math.nan):.4g}, "
            f"HD ratio={row.get('mnln_hd_ratio', math.nan):.4g}"
        )
        display(Markdown(f"**{title}**  \\\n{metric_text}"))
        display(Image(filename=str(row["png_path"])))


def display_multicutoff_results(
    run=RESULT_RUN,
    cutoffs=RESULT_CUTOFFS,
    output_glob=RESULT_OUTPUT_GLOB,
    output_globs=RESULT_OUTPUT_GLOBS,
    result_families=RESULT_FAMILIES,
    mode=RANKING_MODE,
    top_n=TOP_N,
):
    print(f"Base directory: {Path(BASE_DIR)}")
    print(f"Result families: {result_families}")
    print(f"Run filter: {run}")
    print(f"Cutoff filter: {cutoffs}")
    print(f"Ranking mode: {mode} - {RANKING_DESCRIPTIONS.get(mode, 'custom')}")

    roots = discover_result_roots(base_dir=BASE_DIR, output_glob=output_glob, output_globs=output_globs, cutoffs=cutoffs, result_families=result_families)
    if roots:
        print("Result roots found:")
        for root in roots:
            print(f"  family={root['result_family']} cutoff={root['cutoff_gyr']:g}: {root['result_root']}")
    else:
        print("No matching result roots found.")

    metrics = load_multicutoff_metric_results(
        base_dir=BASE_DIR,
        output_glob=output_glob,
        output_globs=output_globs,
        cutoffs=cutoffs,
        run=run,
        expts=RESULT_EXPTS,
        result_families=result_families,
    )
    metrics = filter_metric_results(metrics)
    if metrics.empty:
        print("No metrics CSV rows found after filters.")
        return metrics, pd.DataFrame()

    print(f"Metric rows after filters: {len(metrics)}")
    print(f"Experiments found: {sorted(metrics['expt'].dropna().astype(int).unique())}")
    print(f"Cutoffs found: {format_cutoff_list(metrics['cutoff_gyr'])}")
    display_highdim_coverage(metrics)

    ranked = rank_metric_results(metrics, mode=mode)
    table_columns = STABLE_RESULT_COLUMNS if mode == "stable_cutoff" else TOP_RESULT_COLUMNS
    display_table("Top Results", ranked, columns=table_columns, max_rows=top_n)

    best_expt = best_per_group(metrics, ["run", "expt"], mode=mode)
    display_table("Best Result Per Experiment", rank_metric_results(best_expt, "mnln_p" if mode == "stable_cutoff" else mode), columns=TOP_RESULT_COLUMNS, max_rows=None)

    best_cutoff = best_per_group(metrics, ["run", "cutoff_gyr"], mode=mode)
    display_table("Best Result Per Cutoff", rank_metric_results(best_cutoff, "mnln_p" if mode == "stable_cutoff" else mode), columns=TOP_RESULT_COLUMNS, max_rows=None)

    pivot = make_cutoff_expt_pivot(metrics, value=PIVOT_VALUE)
    display(Markdown(f"### Cutoff by Experiment Pivot: `{PIVOT_VALUE}`"))
    if pivot.empty:
        print(f"Cannot build pivot because `{PIVOT_VALUE}` is unavailable.")
    else:
        display(pivot)

    if SHOW_TOP_PNGS:
        display_png_gallery(ranked, max_images=MAX_GALLERY_IMAGES)

    return metrics, ranked


In [ ]:
all_metric_results, ranked_metric_results = display_multicutoff_results()


## Generate high-dimensional jobs

Run the 2D results viewer first if `HD_MODE = "top_2d"`; it uses `ranked_metric_results` to choose the top run/expt/cutoff combinations and writes HD jobs into separate `_hd` result folders. Set `HD_MODE = "full_sweep"` to generate HD jobs for every configured cutoff and `HD_RUN_EXPTS` entry.


In [ ]:
HD_MODE = "top_2d"      # "top_2d" or "full_sweep"
HD_TOP_N = 20
HD_CUTOFFS = RESULT_CUTOFFS
HD_RUN_EXPTS = None       # None uses RUN_EXPTS for full_sweep. Example: ["10:1-18"]
HD_OVERLAY_GROUPS = OVERLAY_GROUPS
HD_REQUIRE_HIGHDIM = True
HD_SUBMIT_JOBS = False

HD_OUTPUT_SUFFIX = "_hd"
HD_JOB_PREFIX_TEMPLATE = "plot_metrics_hd_tsm_{cutoff_label}_"
HD_SLURM_OVERRIDES = {"mem": "100G", "time": "8:00:00"}


def cutoff_label(cutoff):
    cutoff = float(cutoff)
    text = f"{cutoff:.1f}" if cutoff.is_integer() else f"{cutoff:g}"
    return text.replace(".", "p")


def hd_output_dir(cutoff):
    return BASE_DIR / f"static_umap_metrics_tsm_le_{cutoff_label(cutoff)}gyr{HD_OUTPUT_SUFFIX}"


def hd_job_prefix(cutoff):
    return HD_JOB_PREFIX_TEMPLATE.format(cutoff_label=cutoff_label(cutoff))


def run_expt_spec(run, expts):
    expt_text = ",".join(str(int(expt)) for expt in sorted(set(expts)))
    return f"{int(run)}:{expt_text}"


def top_2d_run_expts_by_cutoff(ranked, top_n=HD_TOP_N):
    if ranked is None or ranked.empty:
        raise ValueError("No ranked 2D results found. Run `all_metric_results, ranked_metric_results = display_multicutoff_results()` first, or set HD_MODE='full_sweep'.")
    work = ranked.copy()
    if "rank" in work.columns:
        work = work.sort_values("rank")
    if "result_family" in work.columns:
        work = work[work["result_family"].fillna("2d").eq("2d")]
    if "cutoff_gyr" in work.columns:
        work["_hd_cutoff_gyr"] = work["cutoff_gyr"]
    else:
        work["_hd_cutoff_gyr"] = pd.NA
    if "best_cutoff_gyr" in work.columns:
        work["_hd_cutoff_gyr"] = work["_hd_cutoff_gyr"].fillna(work["best_cutoff_gyr"])
    work = work.dropna(subset=["run", "expt", "_hd_cutoff_gyr"])
    work = work.head(int(top_n))
    if work.empty:
        raise ValueError("The top-ranked table did not contain any 2D run/expt/cutoff rows for HD targeting.")

    plan = {}
    for _, row in work.iterrows():
        cutoff = float(row["_hd_cutoff_gyr"])
        run = int(row["run"])
        expt = int(row["expt"])
        plan.setdefault(cutoff, {}).setdefault(run, set()).add(expt)
    return {
        cutoff: [run_expt_spec(run, expts) for run, expts in sorted(run_map.items())]
        for cutoff, run_map in sorted(plan.items())
    }


def full_sweep_run_expts_by_cutoff():
    run_expts = RUN_EXPTS if HD_RUN_EXPTS is None else HD_RUN_EXPTS
    return {float(cutoff): list(run_expts) for cutoff in HD_CUTOFFS}


def resolve_hd_run_expts_by_cutoff():
    if HD_MODE == "top_2d":
        return top_2d_run_expts_by_cutoff(globals().get("ranked_metric_results"), top_n=HD_TOP_N)
    if HD_MODE == "full_sweep":
        return full_sweep_run_expts_by_cutoff()
    raise ValueError("HD_MODE must be 'top_2d' or 'full_sweep'")


def build_hd_generator_command(action, cutoff, run_expts, *, dry_run=False):
    slurm_overrides = {**SLURM_OVERRIDES, **HD_SLURM_OVERRIDES}
    cmd = [
        GENERATOR_PYTHON,
        GENERATOR,
        action,
        "--profile",
        PROFILE,
        "--base-directory",
        BASE_DIR,
        "--output-dir",
        hd_output_dir(cutoff),
        "--analysis-script",
        ANALYSIS_SCRIPT,
        "--python-executable",
        JOB_PYTHON,
        "--job-prefix",
        hd_job_prefix(cutoff),
        "--catalog-key",
        CATALOG_KEY,
        "--n-permutations",
        N_PERMUTATIONS,
        "--min-cluster-size",
        MIN_CLUSTER_SIZE,
        "--seed",
        SEED,
        "--dpi",
        DPI,
        "--time-since-merger-max-gyr",
        cutoff,
        "--include-highdim",
    ]
    if HD_REQUIRE_HIGHDIM:
        cmd.append("--require-highdim")
    for spec in run_expts:
        cmd.extend(["--run-expts", spec])
    for group in HD_OVERLAY_GROUPS:
        cmd.extend(["--overlay-group", group])
    for key, value in slurm_overrides.items():
        cmd.extend(["--slurm", f"{key}={value}"])
    if DENSITY:
        cmd.append("--density")
    if LOG_COLORBAR:
        cmd.append("--log-colorbar")
    if not SHOW_LEGEND:
        cmd.append("--no-show-legend")
    if not SUPPRESS_LOGS:
        cmd.append("--no-suppress-logs")
    if SETUP_LINES is not None:
        if len(SETUP_LINES) == 0:
            cmd.append("--no-default-setup")
        else:
            for line in SETUP_LINES:
                cmd.extend(["--setup-line", line])
    elif NO_DEFAULT_SETUP:
        cmd.append("--no-default-setup")
    if dry_run:
        cmd.append("--dry-run")
    return cmd


def build_hd_generator_commands(action, *, dry_run=False):
    commands = []
    plan = resolve_hd_run_expts_by_cutoff()
    print(f"HD mode: {HD_MODE}")
    for cutoff, run_expts in plan.items():
        print(f"cutoff={cutoff:g} Gyr | output={hd_output_dir(cutoff)} | prefix={hd_job_prefix(cutoff)} | run_expts={run_expts or '[generator defaults]'}")
        commands.append(build_hd_generator_command(action, cutoff, run_expts, dry_run=dry_run))
    return commands


def run_hd_generator_commands(action, *, dry_run=False, check=True):
    results = []
    for cmd in build_hd_generator_commands(action, dry_run=dry_run):
        results.append(run_command(cmd, check=check))
    return results


### Preview HD plan

This prints the generator plan for each targeted cutoff without writing scripts.


In [ ]:
run_hd_generator_commands("print-plan")


### Generate HD Slurm scripts

This writes HD scripts with separate output directories and job prefixes.


In [ ]:
run_hd_generator_commands("write")


### Dry-run HD submission

This prints the `sbatch` commands for already-generated HD scripts.


In [ ]:
run_hd_generator_commands("submit-existing", dry_run=True)


### Submit HD jobs

Real submission is guarded by `HD_SUBMIT_JOBS`.


In [ ]:
if HD_SUBMIT_JOBS:
    run_hd_generator_commands("submit-existing")
else:
    print("HD_SUBMIT_JOBS is False; not submitting.")
